# 가까운 IC + 직선거리 변수 값 저장

(1) 라이브러리 임포트 및 API 키 설정

In [1]:
from dotenv import load_dotenv
import os
import pandas as pd
import requests
import time

# .env 파일에서 카카오 API 키 불러오기
load_dotenv()
KAKAO_API_KEY = os.environ.get('KAKAO_API_KEY')

(2) 데이터 로드 

In [2]:
df = pd.read_csv('new_city.csv', encoding='utf-8-sig')
print("데이터 크기:", df.shape)
df.head()

데이터 크기: (90761, 34)


,도시명,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,가장 가까운 IC와의 거리,계약연도,발표후경과년수,CPI,서울도심거리,단지별_세대수,도시별_세대수,기차역까지의거리,기차역이름,지하철호선개수
0,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201101,11,"30,200",...,2.795,2011,8,89.850,28.412513,692.0,27992.0,8.0317,인천역,1
1,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.7379,201012,20,"30,200",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1
2,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201012,11,"28,500",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1
3,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201011,18,"30,600",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1
4,청라,인천광역시 서구 청라동,130-2,130,2,청라웰카운티1차,84.4963,201011,16,"27,476",...,2.795,2010,7,86.373,28.412513,692.0,27992.0,8.0317,인천역,1


(3) 좌표 변환 함수

In [3]:
# 주소 문자열을 입력받아 위도, 경도 반환
def get_coordinates(address):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}
    params = {'query': address}
    response = requests.get(url, headers=headers, params=params)
    result = response.json()
    if result['documents']:
        lat = float(result['documents'][0]['y'])
        lng = float(result['documents'][0]['x'])
        return lat, lng
    else:
        return None, None

(4) 주변 고속도로 IC 탐색 함수

In [4]:
# 좌표 기준으로 반경 10km 내 가장 가까운 고속도로 IC 이름과 거리(m) 반환
def search_nearest_ic(lat, lng, radius=10000):
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    headers = {'Authorization': f'KakaoAK {KAKAO_API_KEY}'}
    params = {
        'query':  '고속도로 IC',
        'x':      lng,
        'y':      lat,
        'radius': radius,
        'sort':   'distance',
        'size':   1
    }
    response = requests.get(url, headers=headers, params=params)
    result = response.json()
    if result['documents']:
        place = result['documents'][0]
        return {
            'name':     place['place_name'],
            'distance': int(place['distance'])
        }
    else:
        return None

(5) 단지별 좌표 추출

In [5]:
# 중복 제거한 단지별로 좌표 API 호출
unique_complexes = df[['단지명', '시군구', '도로명']].drop_duplicates().reset_index(drop=True)
print(f"고유 단지 수: {len(unique_complexes)}")

coord_dict = {}

for _, row in unique_complexes.iterrows():
    apt_name = row['단지명']

    # 1차: 도로명
    full_address = f"{row['시군구']} {row['도로명']}"
    lat, lng = get_coordinates(full_address)

    # 2차: 단지명
    if lat is None:
        full_address = f"{row['시군구']} {apt_name}"
        lat, lng = get_coordinates(full_address)

    # 3차: 괄호 제거 후 단지명
    if lat is None:
        clean_name = apt_name.split('(')[0].strip()
        full_address = f"{row['시군구']} {clean_name}"
        lat, lng = get_coordinates(full_address)

    coord_dict[apt_name] = (lat, lng)
    print(f"  {apt_name}: ({lat}, {lng})")
    time.sleep(0.1)

고유 단지 수: 215
  청라웰카운티1차: (37.53915215032671, 126.6579507991236)
  힐데스하임: (37.528482962173534, 126.65162000586596)
  호반베르디움앤영무예다음: (37.5374052810806, 126.657385130548)
  호반베르디움(116-6): (37.5372412737735, 126.653055810092)
  청라하우스토리커낼뷰: (37.53492771899579, 126.64165844218626)
  서해그랑블: (37.53103297214876, 126.65629649817643)
  청라엘에이치: (37.52848927529058, 126.64676887354958)
  청라우미린: (37.52885888492079, 126.62719519355979)
  청라메이루즈 커낼파크뷰: (37.535232258095604, 126.65343282993709)
  청라자이: (37.528460211394005, 126.65895106788645)
  청라지구중흥S-CLASS2단지: (37.53494969407704, 126.65570828855589)
  청라제일풍경채: (37.53976098795881, 126.64685148525466)
  청라SKVIEW: (37.5257533008036, 126.629821639793)
  청라힐스테이트커낼뷰: (37.534910435002395, 126.64368828777843)
  청라반도유보라: (37.52518430936221, 126.62766825054952)
  청라29블럭호반베르디움: (37.527521215340926, 126.63859975379631)
  청라한화꿈에그린: (37.53780174480017, 126.63865197483484)
  청라한라비발디: (37.54032215960669, 126.63846556085076)
  청라힐스테이트: (37.52598685490953, 126.6284493716

(6) 가장 가까운 IC 탐색

In [6]:
# 단지별 좌표로 가장 가까운 IC 탐색하고 결과를 딕셔너리 저장
ic_name_dict = {}
ic_dist_dict = {}

for apt_name, (lat, lng) in coord_dict.items():
    if lat is None:
        ic_name_dict[apt_name] = None
        ic_dist_dict[apt_name] = None
        continue

    ic = search_nearest_ic(lat, lng)
    time.sleep(0.1)

    if ic is not None:
        ic_name_dict[apt_name] = ic['name']
        ic_dist_dict[apt_name] = ic['distance']
    else:
        ic_name_dict[apt_name] = None
        ic_dist_dict[apt_name] = None

    print(f"  {apt_name} → {ic_name_dict[apt_name]} {ic_dist_dict[apt_name]}m")

  청라웰카운티1차 → 청라IC 2795m
  힐데스하임 → 남청라IC 2758m
  호반베르디움앤영무예다음 → 청라IC 2995m
  호반베르디움(116-6) → 청라IC 3110m
  청라하우스토리커낼뷰 → 남청라IC 2649m
  서해그랑블 → 서인천IC 3142m
  청라엘에이치 → 남청라IC 2412m
  청라우미린 → 남청라IC 1570m
  청라메이루즈 커낼파크뷰 → 청라IC 3313m
  청라자이 → 서인천IC 2848m
  청라지구중흥S-CLASS2단지 → 청라IC 3292m
  청라제일풍경채 → 북청라IC 3017m
  청라SKVIEW → 남청라IC 1274m
  청라힐스테이트커낼뷰 → 남청라IC 2748m
  청라반도유보라 → 남청라IC 1170m
  청라29블럭호반베르디움 → 남청라IC 1823m
  청라한화꿈에그린 → 북청라IC 2751m
  청라한라비발디 → 북청라IC 2504m
  청라힐스테이트 → 남청라IC 1270m
  호반베르디움 → 서인천IC 3040m
  청라동양엔파트 → 남청라IC 1140m
  한일베라체 → 남청라IC 2298m
  청라 엑슬루타워 → 남청라IC 2996m
  NPART → 남청라IC 2412m
  청라지구중흥S-CLASS1단지 → 청라IC 2951m
  청라롯데캐슬 → 남청라IC 2750m
  청라반도유보라2.0 → 남청라IC 1046m
  청라린스트라우스 → 남청라IC 3098m
  청라한양수자인 → 남청라IC 1458m
  청라더샵레이크파크 → 남청라IC 2127m
  대우푸르지오 → 남청라IC 2518m
  청라골드클래스 → 남청라IC 1525m
  청라디이스트 → 남청라IC 1314m
  청라웰카운티 2차 → 서인천IC 3153m
  청라골드클래스커낼웨이 → 남청라IC 3096m
  청라제일풍경채2차에듀앤파크 → 북청라IC 3176m
  청라모아미래도 → 북인천IC 1900m
  청라국제업무단지센텀대광로제비앙 → 북인천IC 1963m
  청라센트럴에일린의뜰 → 남청라IC 2943m
  청라국제금융

(7) 컬럼 추가 및 CSV 저장

In [7]:
# 단지명 기준으로 컬럼 추가 후 CSV 저장 (거리 단위 m -> km로 변환)
df['가장 가까운 고속도로 IC'] = df['단지명'].apply(lambda x: ic_name_dict.get(x))
df['가장 가까운 IC와의 거리'] = df['단지명'].apply(
    lambda x: round(ic_dist_dict.get(x) / 1000, 3) if ic_dist_dict.get(x) is not None else None)

print(df[['단지명', '가장 가까운 고속도로 IC', '가장 가까운 IC와의 거리']].drop_duplicates('단지명').to_string())

df.to_csv('new_city.csv', index=False, encoding='utf-8-sig')
print("최종 데이터 크기:", df.shape)

                        단지명 가장 가까운 고속도로 IC  가장 가까운 IC와의 거리
0                  청라웰카운티1차           청라IC           2.795
14                    힐데스하임          남청라IC           2.758
15             호반베르디움앤영무예다음           청라IC           2.995
16            호반베르디움(116-6)           청라IC           3.110
17               청라하우스토리커낼뷰          남청라IC           2.649
18                    서해그랑블          서인천IC           3.142
20                   청라엘에이치          남청라IC           2.412
21                    청라우미린          남청라IC           1.570
40             청라메이루즈 커낼파크뷰           청라IC           3.313
44                     청라자이          서인천IC           2.848
53         청라지구중흥S-CLASS2단지           청라IC           3.292
128                 청라제일풍경채          북청라IC           3.017
129                청라SKVIEW          남청라IC           1.274
130              청라힐스테이트커낼뷰          남청라IC           2.748
132                 청라반도유보라          남청라IC           1.170
134            청라29블럭호반베르디움          남청라IC           1.8